# CÂU 9: TỶ LỆ TRẢ HÀNG CAO NHẤT THEO KÍCH THƯỚC

**1. Yêu cầu bài toán:**
Tìm kích thước sản phẩm (S, M, L, XL) có tỷ lệ trả hàng cao nhất.
* Tỷ lệ trả hàng = Số bản ghi trong `returns` / Số dòng trong `order_items`. (Cần join với bảng `products` để lấy kích thước).

**2. Phương pháp tiếp cận & Suy luận:**
* **Bước 1:** Tải dữ liệu từ 3 file: `returns.csv`, `order_items.csv` và `products.csv`.
* **Bước 2 (Tử số):** Nối bảng `returns` với `products` theo `product_id`. Sau đó đếm số lượng dòng cho từng kích thước (size).
* **Bước 3 (Mẫu số):** Nối bảng `order_items` với `products` theo `product_id`. Sau đó đếm số lượng dòng cho từng kích thước.
* **Bước 4:** Lấy Tử số chia cho Mẫu số để ra tỷ lệ trả hàng cho từng size, sau đó sắp xếp giảm dần để tìm top 1.

**3. Lý do chọn phương pháp:**
* **Sử dụng `value_counts()` sau khi Join:** Thay vì viết các vòng lặp đếm phức tạp, ta join bảng để gắn mác size cho từng lượt trả hàng và từng lượt mua hàng. Sau đó dùng `value_counts()` đếm tổng số lần xuất hiện của mỗi size một cách cực kỳ nhanh chóng.
* **Tính toán Series (Vectorized division):** Pandas cho phép lấy trực tiếp chuỗi số đếm của returns (Series) chia cho chuỗi số đếm của order_items. Nó sẽ tự động khớp các size (S chia cho S, M chia cho M...) mà không sợ bị lệch dòng.

In [4]:
# Import thư viện
import pandas as pd
from google.colab import drive

# Kết nối với Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Thực hiện Bước 1, 2 & 3: Đếm số lượng Trả hàng và Bán ra theo Size**

Ta sẽ load 3 bảng dữ liệu lên, thực hiện phép nối và đếm riêng biệt tử số và mẫu số.

In [5]:
# 1. Khai báo đường dẫn
path_ret = '/content/drive/MyDrive/datathon-2026-round-1/returns.csv'
path_items = '/content/drive/MyDrive/datathon-2026-round-1/order_items.csv'
path_prod = '/content/drive/MyDrive/datathon-2026-round-1/products.csv'

# 2. Đọc dữ liệu
df_returns = pd.read_csv(path_ret)
df_items = pd.read_csv(path_items)
df_products = pd.read_csv(path_prod)

# Giả định cột chứa kích thước trong bảng products tên là 'size'
# Tử số: Nối returns với products và đếm theo size
returns_merged = pd.merge(df_returns, df_products[['product_id', 'size']], on='product_id', how='inner')
return_counts = returns_merged['size'].value_counts()

# Mẫu số: Nối order_items với products và đếm theo size
items_merged = pd.merge(df_items, df_products[['product_id', 'size']], on='product_id', how='inner')
item_counts = items_merged['size'].value_counts()

/tmp/ipykernel_2650/2317078017.py:8: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_items = pd.read_csv(path_items)


**Thực hiện Bước 4: Tính Tỷ lệ và Xếp hạng**

Chia tử số cho mẫu số và hiển thị kết quả để tìm ra size có tỷ lệ trả về (return rate) cao nhất.

In [6]:
# Tính tỷ lệ trả hàng (chia tự động theo index là tên các size)
return_rates = (return_counts / item_counts).sort_values(ascending=False)

# Chỉ lọc ra 4 size theo yêu cầu đề bài để nhìn cho gọn
sizes_to_check = ['S', 'M', 'L', 'XL']
final_rates = return_rates[return_rates.index.isin(sizes_to_check)].sort_values(ascending=False)

print("Tỷ lệ trả hàng theo từng kích thước (S, M, L, XL):")
print(final_rates)

# Chốt kết quả cao nhất
highest_size = final_rates.idxmax()
highest_rate = final_rates.max()

print(f"\n--> Kích thước có tỷ lệ trả hàng cao nhất là: '{highest_size}' (với tỷ lệ {highest_rate:.4f})")

Tỷ lệ trả hàng theo từng kích thước (S, M, L, XL):
size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
Name: count, dtype: float64

--> Kích thước có tỷ lệ trả hàng cao nhất là: 'S' (với tỷ lệ 0.0565)


**KẾT LUẬN CUỐI CÙNG:**
Dựa trên kết quả tính toán phân chia giữa bảng trả hàng và bảng đơn hàng, kích thước sản phẩm có tỷ lệ trả hàng cao nhất là **S** (với tỷ lệ khoảng 0.0565).

Đối chiếu với các phương án của đề bài:

A) S

B) M

C) L

D) XL

**=> Đáp án được chọn là: A**